# AgentRegistry, end to end: scaffold a dice agent → run it on kagent **and** AWS Bedrock AgentCore

**Persona:** a **developer**. Run one cell at a time like a terminal. We scaffold a dice agent with `arctl`, run it locally, publish it, then deploy **the same agent** to two runtimes — Solo Enterprise for **kagent** (local kind) and **AWS Bedrock AgentCore** — by changing one line, the Deployment's `runtimeRef`.

> **Kernel:** pick **Bash** (top-right). Engineer setup lives in `setup/` (see `setup/README.md`).

## Connect to the platform

Loads creds, puts `arctl` on the path, mints a catalog token.

In [ ]:
[ -d agentregistry-agentcore-kind ] && cd agentregistry-agentcore-kind; source setup/scripts/connect.sh

## 1. Create a new agent project

Scaffolds the dice agent into `agentdemo/`. **Open it in the Explorer** to walk through `roll_die` / `check_prime`.

In [ ]:
rm -rf agentdemo; arctl init agent agentdemo --framework adk --language python --model-provider anthropic --model-name claude-haiku-4-5

## 2. Walk through the dice agent

In [ ]:
cat agentdemo/agentdemo/agent.py

## 3. Build the agent image

In [ ]:
arctl build ./agentdemo

Run it locally in an interactive chat (use a **terminal** — it's interactive):

```sh
arctl run ./agentdemo
```

## 4. Publish to the catalog

In [ ]:
arctl build ./agentdemo --push

In [ ]:
arctl apply -f agentdemo/agent.yaml

In [ ]:
arctl get agent agentdemo

## 5. Deploy the agent onto kagent (runtime #1)

`runtimeRef: kind-kagent` is **the one line that changes for AWS later.**

In [ ]:
envsubst < setup/yaml/deploy-kagent.yaml | arctl apply -f -

In [ ]:
arctl get deployments

## 6. Talk to the dice agent — through real OIDC

Mints a Keycloak token for **alice** and sends an A2A message; watch it call `roll_die` then `check_prime`.

In [ ]:
./setup/scripts/ask.sh "Roll a 20-sided die and tell me whether the result is a prime number."

---
# The punchline: the **same** agent on AWS Bedrock AgentCore (runtime #2)

Identical agent, deployed to AWS — native Bedrock Claude via the AWS role. **Needs an AWS account; skip for a local-only demo.**

## 7. Sign in to AWS

In [ ]:
source setup/scripts/aws-login.sh

## 8. Deploy the same agent to AgentCore

One command: multi-cloud patch, cross-account role, `BedrockAgentCore` runtime, image+source push, deploy, wait for READY.

In [ ]:
./setup/scripts/agentcore-deploy.sh

## 9. Test the dice agent on AgentCore

In [ ]:
./setup/scripts/ac-invoke.sh "Roll a 20-sided die and tell me whether the result is a prime number."

**The takeaway:** one agent, published once, ran unchanged on kagent *and* AWS Bedrock AgentCore.

## Reset / teardown

```sh
./setup/scripts/reset.sh      # back to start
./setup/scripts/cleanup.sh    # full teardown
```